# ROC 曲线与 AUC：阈值无关的评估
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：12_模型评估与选择 | 源文件：ROC曲线.py | 核心功能：ROC 曲线、AUC 计算、多分类 ROC

## 概述

ROC 曲线画出不同阈值下 TPR（真正率）vs FPR（假正率）。AUC（曲线下面积）是一个阈值无关的综合指标。AUC=1 完美，AUC=0.5 随机。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
# --- 导入库 ---
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score
)

## 数学原理

### 1. ROC 曲线的构建

ROC（Receiver Operating Characteristic）曲线以**假正率（FPR）**为横轴、**真正率（TPR）**为纵轴：

$$\text{TPR} = \frac{TP}{TP + FN} = \text{Recall}$$

$$\text{FPR} = \frac{FP}{FP + TN}$$

通过改变分类阈值 $\theta$（从 $1$ 到 $0$），每个 $\theta$ 对应一个 (FPR, TPR) 点，连成曲线。

### 2. AUC（曲线下面积）

$$\text{AUC} = \int_0^1 \text{TPR}(\text{FPR}^{-1}(t)) \, dt$$

离散近似：

$$\text{AUC} = \sum_{i=1}^{n-1} \frac{(\text{FPR}_{i+1} - \text{FPR}_i)(\text{TPR}_{i+1} + \text{TPR}_i)}{2}$$

AUC 的概率解释：随机选一个正样本和一个负样本，分类器给正样本打分高于负样本的概率。

$$\text{AUC} = P(\hat{p}(x^+) > \hat{p}(x^-))$$

### 3. AUC 的解读

| AUC 范围 | 模型质量 |
|----------|----------|
| 0.9 - 1.0 | 优秀 |
| 0.8 - 0.9 | 良好 |
| 0.7 - 0.8 | 一般 |
| 0.5 - 0.7 | 较差 |
| 0.5 | 随机猜测（对角线） |

### 4. 多分类 ROC（One-vs-Rest）

对 $C$ 个类别，每个类别 $c$ 构建二分类问题：

$$\text{TPR}_c = \frac{TP_c}{TP_c + FN_c}, \quad \text{FPR}_c = \frac{FP_c}{FP_c + TN_c}$$

宏平均 AUC：$\text{AUC}_{macro} = \frac{1}{C}\sum_{c=1}^{C}\text{AUC}_c$

### 5. Precision-Recall 曲线

当类别**严重不平衡**时（正样本远少于负样本），PR 曲线比 ROC 更有信息量。

$$\text{PR 曲线}: (R, P) = \left(\frac{TP}{TP+FN}, \frac{TP}{TP+FP}\right)$$

**AP（Average Precision）**：PR 曲线下面积的近似

$$\text{AP} = \sum_{k}(R_k - R_{k-1}) P_k$$

### 6. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `roc_curve(y_true, y_score)` | 返回 FPR, TPR, 阈值数组 |
| `auc(fpr, tpr)` | ROC 曲线下面积 |
| `roc_auc_score(y_true, y_score)` | 直接计算 AUC |
| `precision_recall_curve(y_true, y_score)` | 返回 P, R, 阈值 |
| `average_precision_score(y_true, y_score)` | AP 值 |
| `label_binarize(y, classes=...)` | one-hot 编码用于多分类 ROC |

### 1. 二分类ROC曲线

在分类任务上训练模型并评估性能。

In [ ]:
print("=" * 60)
print("1. 二分类ROC曲线")
print("=" * 60)

X_binary, y_binary = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_classes=2, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_binary, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

# 两个模型对比
models = {
    '逻辑回归': LogisticRegression(max_iter=1000, random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]

    # 计算ROC曲线
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    print(f"\n{name}:")
    print(f"  AUC = {roc_auc:.4f}")

    # 打印关键阈值点
    print(f"  {'阈值':>8} {'FPR':>8} {'TPR':>8} {'1-特异度':>8} {'灵敏度':>8}")
    # 选择5个代表性阈值点
    n_points = len(thresholds)
    indices = np.linspace(0, n_points - 1, 5, dtype=int)
    for idx in indices:
        print(f"  {thresholds[idx]:>8.4f} {fpr[idx]:>8.4f} {tpr[idx]:>8.4f} {fpr[idx]:>8.4f} {tpr[idx]:>8.4f}")

    # 找到最优阈值 (Youden's J statistic)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_threshold = thresholds[best_idx]
    print(f"  最优阈值 (Youden J): {best_threshold:.4f}")
    print(f"  对应TPR: {tpr[best_idx]:.4f}, FPR: {fpr[best_idx]:.4f}")

### 2. 多分类ROC曲线 (One-vs-Rest)

在分类任务上训练模型并评估性能。

In [ ]:
print("\n" + "=" * 60)
print("2. 多分类ROC曲线 (One-vs-Rest)")
print("=" * 60)

X_multi, y_multi = make_classification(
    n_samples=600, n_features=10, n_informative=5,
    n_classes=4, n_clusters_per_class=1, random_state=42
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42, stratify=y_multi
)

model_multi = LogisticRegression(max_iter=1000, random_state=42)
model_multi.fit(X_train_m, y_train_m)
y_proba_m = model_multi.predict_proba(X_test_m)

n_classes = len(np.unique(y_multi))
class_names = [f"类{i}" for i in range(n_classes)]

print(f"类别数: {n_classes}")
print()

# One-vs-Rest: 每个类别 vs 其他所有类别
print("One-vs-Rest ROC曲线:")
for i in range(n_classes):
    fpr_i, tpr_i, _ = roc_curve(y_test_m == i, y_proba_m[:, i])
    auc_i = auc(fpr_i, tpr_i)
    print(f"  {class_names[i]} vs 其他: AUC = {auc_i:.4f}")

# 整体多分类AUC
auc_ovr = roc_auc_score(y_test_m, y_proba_m, multi_class='ovr', average='macro')
auc_ovo = roc_auc_score(y_test_m, y_proba_m, multi_class='ovo', average='macro')
print(f"\n  宏平均AUC (One-vs-Rest): {auc_ovr:.4f}")
print(f"  宏平均AUC (One-vs-One):  {auc_ovo:.4f}")

### 3. Precision-Recall曲线

运行 3. Precision-Recall曲线 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("3. Precision-Recall曲线")
print("=" * 60)

# 使用二分类数据
model_pr = LogisticRegression(max_iter=1000, random_state=42)
model_pr.fit(X_train, y_train)
y_proba_pr = model_pr.predict_proba(X_test)[:, 1]

precision, recall, thresholds_pr = precision_recall_curve(y_test, y_proba_pr)
ap = average_precision_score(y_test, y_proba_pr)

print(f"平均精度 (Average Precision): {ap:.4f}")
print(f"\nPR曲线关键点:")
print(f"  {'阈值':>8} {'精确率':>8} {'召回率':>8}")

# 选择代表性点
n_pr = len(thresholds_pr)
indices_pr = np.linspace(0, n_pr - 1, 6, dtype=int)
for idx in indices_pr:
    if idx < len(thresholds_pr):
        print(f"  {thresholds_pr[idx]:>8.4f} {precision[idx]:>8.4f} {recall[idx]:>8.4f}")

# 在不同召回率水平下的精确率
print(f"\n不同召回率水平下的精确率:")
for target_recall in [0.5, 0.6, 0.7, 0.8, 0.9]:
    idx = np.searchsorted(-recall, -target_recall)
    if idx < len(precision):
        print(f"  召回率>={target_recall:.1f}时, 最高精确率: {precision[idx]:.4f}")

### 4. ROC vs PR 曲线对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("4. ROC曲线 vs PR曲线")
print("=" * 60)

# 不同正负比例下的对比
print(f"\n原始数据正负比例: {np.bincount(y_test)}")
print(f"  正类比例: {y_test.mean():.4f}")

# 模拟不平衡数据
X_imb, y_imb = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_classes=2, weights=[0.95, 0.05], random_state=42
)

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb
)

model_imb = LogisticRegression(max_iter=1000, random_state=42)
model_imb.fit(X_train_imb, y_train_imb)
y_proba_imb = model_imb.predict_proba(X_test_imb)[:, 1]

fpr_imb, tpr_imb, _ = roc_curve(y_test_imb, y_proba_imb)
precision_imb, recall_imb, _ = precision_recall_curve(y_test_imb, y_proba_imb)

auc_roc_imb = auc(fpr_imb, tpr_imb)
ap_imb = average_precision_score(y_test_imb, y_proba_imb)

print(f"\n不平衡数据 (正类仅5%):")
print(f"  AUC-ROC: {auc_roc_imb:.4f}")
print(f"  Average Precision: {ap_imb:.4f}")
print(f"  说明: 不平衡时ROC可能过于乐观，PR曲线更真实反映模型对少数类的识别能力")

### 总结

运行 总结 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("总结")
print("=" * 60)
print("""
ROC曲线:
# --- 继续 ---
  - 横轴: FPR = FP / (FP + TN)，假阳性率
  - 纵轴: TPR = TP / (TP + FN)，真阳性率（召回率）
  - AUC = 1.0: 完美分类器
  - AUC = 0.5: 等同随机猜测
  - 适合类别平衡的场景

Precision-Recall曲线:
  - 横轴: 召回率（灵敏度）
  - 纵轴: 精确率
  - 适合类别不平衡的场景
  - Average Precision: PR曲线下的面积

多分类:
  - 使用One-vs-Rest或One-vs-One策略拆分为多个二分类问题
  - macro: 每个类别平等加权
  - weighted: 按类别样本数加权
""")

## 关键代码解释

```python
from sklearn.metrics import roc_curve, auc
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)
```

## 注意事项

1. 需要概率输出（`predict_proba`），不是类别预测
2. 类别不平衡时 ROC 可能过于乐观，用 PR 曲线
3. 多分类需要 One-vs-Rest 展开

## 总结与延伸

以上代码展示了 **ROC曲线** 的完整流程。

**解读要点**：
- 关注 **ROC 曲线** 下的面积（AUC）
- 观察 **交叉验证** 各折的方差
- 对比不同评估指标的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **PR 曲线**：精确率 vs 召回率，不平衡数据更好
- **ROC 的阈值选择**：Youden's J 统计量
- **多分类 AUC**：macro/micro/weighted 平均
